In [ ]:
import re
from io import StringIO
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

csv_dir = Path("csv")

def read_trials_csv(path):
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r",(?=\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2},)", "\n", text)

    lines = text.splitlines()
    if not lines:
        return pd.DataFrame()

    valid_lines = [lines[0]]
    for line in lines[1:]:
        if line.count(",") >= 24:
            valid_lines.append(line)

    return pd.read_csv(StringIO("\n".join(valid_lines)), on_bad_lines="skip")


items = pd.concat(
    [pd.read_csv(f) for f in sorted(csv_dir.glob("*_items.csv"))],
    ignore_index=True
)
trials = pd.concat(
    [read_trials_csv(f) for f in sorted(csv_dir.glob("*_trials.csv"))],
    ignore_index=True
)
items = items.drop(columns=["chunk_label", "timestamp", "participant"], errors="ignore")


In [ ]:
iex1 = items[(items["experiment"] == "free_recall") & (items["condition"] == "baseline")]
tex1 = trials[(trials["experiment"] == "free_recall") & (trials["condition"] == "baseline")]
iex2 = items[items["condition"] == "unfilled_delay"]
tex2 = trials[trials["condition"] == "unfilled_delay"]
iex3 = items[items["condition"] == "fast_rate"]
tex3 = trials[trials["condition"] == "fast_rate"]
iex4 = items[items["condition"] == "working_memory"]
tex4 = trials[trials["condition"] == "working_memory"]
iex5 = items[items["condition"].str.fullmatch(r"capacity_n\d")]
tex5 = trials[trials["condition"].str.fullmatch(r"capacity_n\d")]
iex6 = items[(items["experiment"] == "serial_recall") & (items["condition"] == "baseline")]
tex6 = trials[(trials["experiment"] == "serial_recall") & (trials["condition"] == "baseline")]
iex7 = items[items["condition"] == "finger_tapping"]
tex7 = trials[trials["condition"] == "finger_tapping"]
iex8 = items[items["condition"] == "articulatory_suppression"]
tex8 = trials[trials["condition"] == "articulatory_suppression"]
iex9 = items[items["condition"] == "real_idioms"]
tex9 = trials[trials["condition"] == "real_idioms"]
iex10 = items[items["condition"] == "fake_idioms"]
tex10 = trials[trials["condition"] == "fake_idioms"]


In [ ]:
serial_curve1 = iex1.groupby("serial_position")["free_recalled"].mean()  # Baseline
serial_curve2 = iex2.groupby("serial_position")["free_recalled"].mean()  # Unfilled Delay
serial_curve3 = iex3.groupby("serial_position")["free_recalled"].mean()  # Fast Rate
serial_curve4 = iex4.groupby("serial_position")["free_recalled"].mean()  # Working Memory

# Graph 1: Baseline vs Unfilled Delay
plt.plot(serial_curve1.index, serial_curve1.values, marker="o", label="Baseline")
plt.plot(serial_curve2.index, serial_curve2.values, marker="o", label="Unfilled Delay")
plt.xlabel("Serial Position")
plt.ylabel("Average Free Recall")
plt.ylim(0, 1)
plt.title("Baseline vs Unfilled Delay")
plt.legend()
plt.show()

# Graph 2: Baseline vs Fast Rate
plt.plot(serial_curve1.index, serial_curve1.values, marker="o", label="Baseline")
plt.plot(serial_curve3.index, serial_curve3.values, marker="o", label="Fast Rate")
plt.xlabel("Serial Position")
plt.ylabel("Average Free Recall")
plt.ylim(0, 1)
plt.title("Baseline vs Fast Rate")
plt.legend()
plt.show()

# Graph 3: Baseline vs Working Memory
plt.plot(serial_curve1.index, serial_curve1.values, marker="o", label="Baseline")
plt.plot(serial_curve4.index, serial_curve4.values, marker="o", label="Working Memory")
plt.xlabel("Serial Position")
plt.ylabel("Average Free Recall")
plt.ylim(0, 1)
plt.title("Baseline vs Working Memory")
plt.legend()
plt.show()

In [ ]:
serial_curve6 = iex6.groupby("serial_position")["serial_correct"].mean()
serial_curve7 = iex7.groupby("serial_position")["serial_correct"].mean()
serial_curve8 = iex8.groupby("serial_position")["serial_correct"].mean()

plt.plot(serial_curve6.index, serial_curve6.values, marker="o", label="baseline")
plt.plot(serial_curve7.index, serial_curve7.values, marker="o", label="finger_tapping")
plt.plot(serial_curve8.index, serial_curve8.values, marker="o", label="articulatory_suppression")

plt.xlabel("Serial Position")
plt.ylabel("Average Serial Recall")
plt.title("Serial Recall Performance by Serial Position")
plt.legend()
plt.show()

In [ ]:
serial_curve9 = iex9.groupby("serial_position")["serial_correct"].mean()
serial_curve10 = iex10.groupby("serial_position")["serial_correct"].mean()

plt.plot(serial_curve9.index, serial_curve9.values, marker="o", label="real_idioms")
plt.plot(serial_curve10.index, serial_curve10.values, marker="o", label="fake_idioms")

plt.xlabel("Serial Position")
plt.ylabel("Average Serial Recall")
plt.title("Serial Recall Performance by Serial Position")
plt.legend()
plt.show()

In [ ]:
# The effect of articulatory suppression
# The (lack of) effect of finger tapping
def conf(data):
    data = data["serial_correct"]
    p = data.sum() / len(data)
    confidence_interval = [
        p - 1.96 * (p * (1 - p) / len(data)) ** 0.5,
        p + 1.96 * (p * (1 - p) / len(data)) ** 0.5
    ]
    return p, confidence_interval[0], confidence_interval[1]

datasets = {"Baseline": iex6, "Finger tapping": iex7, "Articulatory suppression": iex8}

labels, estimates, err_low, err_high = [], [], [], []

for name, d in datasets.items():
    p, lo, hi = conf(d)
    labels.append(name)
    estimates.append(p)
    err_low.append(p - lo)
    err_high.append(hi - p)

fig, ax = plt.subplots(figsize=(5, 3))
ax.errorbar(
    labels, estimates, yerr=[err_low, err_high],
    fmt='o', capsize=6, markersize=8,
    color='black', ecolor='steelblue', elinewidth=2
)
ax.set_ylabel("Proportion correct")
ax.set_title("Serial Recall")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# The effect of chunking
lettercount6 = iex6[iex6['serial_correct'] == 1]["presented_item"].str.len().sum()
experimentcount6 = len(iex6)/12
lettercount9 = iex9[iex9['serial_correct'] == 1]["presented_item"].str.len().sum()
lettercount10 = iex10[iex10['serial_correct'] == 1]["presented_item"].str.len().sum()
experimentcount9 = len(iex9)/12
experimentcount10 = len(iex10)/12
print(lettercount6/experimentcount6, lettercount9/experimentcount9, lettercount10/experimentcount10)

In [ ]:
datasets = {"Real idioms": iex9, "Fake idioms": iex10}

labels, estimates, err_low, err_high = [], [], [], []

for name, d in datasets.items():
    p, lo, hi = conf(d)
    labels.append(name)
    estimates.append(p)
    err_low.append(p - lo)
    err_high.append(hi - p)

fig, ax = plt.subplots(figsize=(3, 3))
ax.errorbar(
    labels, estimates, yerr=[err_low, err_high],
    fmt='o', capsize=6, markersize=8,
    color='black', ecolor='steelblue', elinewidth=2
)
ax.set_ylabel("Proportion correct")
ax.set_title("95% Confidence Intervals")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
err = items[
    (items["condition"] != "working_memory") &
    (~items["condition"].isin(["real_idioms", "fake_idioms"])) &
    (items["substitution"].notna()) &
    (items["substitution"].astype(str).str.strip().str.len() > 0)
].copy()

pair_counts = (
    err.groupby(["presented_item", "substitution"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(pair_counts)

In [ ]:
data = items[
    ~items["condition"].isin([
        "working_memory", "real_idioms", "fake_idioms"
    ])
].copy()

data["group"] = "others"
data.loc[
    data["condition"] == "articulatory_suppression", "group"
] = "suppression"

totals = data.groupby("group").size().reindex(
    ["others", "suppression"], fill_value=0
)

err = data[
    data["substitution"].notna() &
    data["substitution"].astype(str).str.strip().ne("")
].copy()

# Alfabetisk sortering gør Q/M og M/Q til samme par.
pairs = [
    sorted([str(a).strip().upper(), str(b).strip().upper()])
    for a, b in zip(err["presented_item"], err["substitution"])
]
err["presented_item"] = [pair[0] for pair in pairs]
err["substitution"] = [pair[1] for pair in pairs]

comparison = (
    err.groupby(["presented_item", "substitution", "group"])
    .size()
    .unstack("group", fill_value=0)
    .reindex(columns=["others", "suppression"], fill_value=0)
    .rename(columns={
        "others": "others_count",
        "suppression": "suppression_count",
    })
)
comparison.columns.name = None

for group in ["others", "suppression"]:
    comparison[f"{group}_%"] = (
        comparison[f"{group}_count"] / totals[group] * 100
        if totals[group] else float("nan")
    )

comparison["difference_pp"] = (
    comparison["suppression_%"] - comparison["others_%"]
)

comparison = (
    comparison.reset_index()
    .sort_values("difference_pp", ascending=False)
    .reset_index(drop=True)
)

display(comparison.round(3))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(
    1, 2, figsize=(14, 7),
    sharex=True, constrained_layout=True
)
fig.patch.set_facecolor("#F3F6FB")

groups = [
    ("others_%", "Others", "#2563EB"),( "suppression_%", "Articulatory suppression", "#8B5CF6"),
]

max_value = comparison[["suppression_%", "others_%"]].max().max()
x_limit = max_value * 1.25 if max_value > 0 else 1

for ax, (column, title, color) in zip(axes, groups):
    top = (
        comparison[comparison[column] > 0]
        .nlargest(10, column)
        .sort_values(column)
    )

    labels = (
        top["presented_item"].astype(str)
        + " ↔ " 
        + top["substitution"].astype(str)
    )

    bars = ax.barh(
        range(len(top)), top[column],
        color=color, alpha=0.85, height=0.65, zorder=3
    )

    ax.set_yticks(range(len(top)), labels)
    ax.bar_label(bars, fmt="%.2f%%", padding=7, fontsize=10)

    ax.set_title(
        title, fontsize=15, fontweight="bold",
        color="#1E293B", pad=15
    )
    ax.set_xlabel("Andel af alle viste items", fontsize=11)
    ax.set_xlim(0, x_limit)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=100))

    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#E2E8F0")
    ax.tick_params(axis="both", length=0, labelsize=11)

    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle(
    "Top 10 substitutions",
    fontsize=19, fontweight="bold", color="#0F172A"
)

plt.show()

In [ ]:
# Free recall summary used by the two cells below.
# Each trial is treated as one datapoint; databehandling's `trials` DataFrame
# and CSV-loading approach are kept unchanged.
free_conditions = ["baseline", "fast_rate", "unfilled_delay", "working_memory"]

free_trials = trials[
    (trials["experiment"] == "free_recall") &
    (trials["condition"].isin(free_conditions))
].copy()

summary_metrics = {
    "free_accuracy": "mean_free_accuracy",
    "start_accuracy": "mean_start_accuracy",
    "middle_accuracy": "mean_middle_accuracy",
    "end_accuracy": "mean_end_accuracy",
    "primacy_index": "mean_primacy",
    "recency_index": "mean_recency",
}

summary_rows = []

for condition in free_conditions:
    group = free_trials[free_trials["condition"] == condition]

    row = {
        "condition": condition,
        "n_participants": (
            group["participant"].nunique()
            if "participant" in group.columns
            else float("nan")
        ),
        "n_trials": len(group),
    }

    for source_column, output_name in summary_metrics.items():
        values = pd.to_numeric(group[source_column], errors="coerce").dropna()

        mean = values.mean()
        if len(values) > 1:
            sem = stats.sem(values)
            margin = stats.t.ppf(0.975, df=len(values) - 1) * sem
            low = mean - margin
            high = mean + margin
        else:
            low = high = float("nan")

        row[output_name] = mean
        row[f"{output_name}_CI_low"] = low
        row[f"{output_name}_CI_high"] = high

    summary_rows.append(row)

summary = (
    pd.DataFrame(summary_rows)
    .set_index("condition")
    .reindex(free_conditions)
)


labels = {
    "mean_free_accuracy": "Samlet accuracy",
    "mean_start_accuracy": "Start (position 1–6)",
    "mean_middle_accuracy": "Midte (position 7–11)",
    "mean_end_accuracy": "Slut (position 12–16)",
    "mean_primacy": "Primacy (start − midte)",
    "mean_recency": "Recency (slut − midte)",
}

for condition, row in summary.iterrows():
    print(
        f"\n{condition.upper()} — "
        f"{int(row['n_participants'])} deltagere, "
        f"{int(row['n_trials'])} forsøg"
    )

    table = []

    for metric, label in labels.items():
        low = row[f"{metric}_CI_low"]
        high = row[f"{metric}_CI_high"]

# Accuracy: 0 til 1. Primacy/recency: -1 til 1.
        minimum = -1 if metric in ["mean_primacy", "mean_recency"] else 0
        low = max(minimum, low)
        high = min(1, high)

        mean = row[metric] * 100
        low = low * 100
        high = high * 100

        unit = "procentpoint" if metric in [
            "mean_primacy", "mean_recency"
        ] else "%"

        table.append({
            "Mål": label,
            "Gennemsnit": f"{mean:.2f} {unit}",
            "95 % konfidensinterval": f"[{low:.2f}; {high:.2f}]",
        })
    display(pd.DataFrame(table).set_index("Mål"))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

conditions = {
    "baseline": "Baseline",
    "fast_rate": "Fast_rate",
    "unfilled_delay": "Unfilled_delay",
    "working_memory": "Working_memory",
}

metrics = [
    "mean_start_accuracy",
    "mean_middle_accuracy",
    "mean_end_accuracy",
]

labels = ["Position 1–6", "Position 7–11", "Position 12–16"]
colors = ["#2563EB", "#8B5CF6", "#0891B2", "#EA580C"]

fig, axes = plt.subplots(
    2, 2, figsize=(12, 9), sharey=True,
    constrained_layout=True
)
fig.patch.set_facecolor("#F3F6FB")

for ax, (condition, title), color in zip(
    axes.flat, conditions.items(), colors
):
    row = summary.loc[condition]
    x = np.arange(3)

    means = np.array([row[m] for m in metrics])

    lower = np.clip(
        [row[f"{m}_CI_low"] for m in metrics], 0, 1
    )
    upper = np.clip(
        [row[f"{m}_CI_high"] for m in metrics], 0, 1
    )

    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#E2E8F0", linewidth=0.8)

    ax.bar(
        x, means, width=0.55,
        color=color, alpha=0.8, zorder=3
    )

    ax.errorbar(
        x, means,
        yerr=[means - lower, upper - means],
        fmt="none", ecolor="#1E293B",
        elinewidth=1.6, capsize=7, capthick=1.6,
        zorder=4
    )

    ax.axhline(
        row["mean_free_accuracy"],
        color="#64748B", linestyle="--", linewidth=1.5
    )

    for position, value in zip(x, means):
        ax.text(
            position, value / 2,
            f"{value:.1%}",
            ha="center", va="center",
            color="white", fontsize=12, fontweight="bold"
        )

    ax.set_title(
        title,
        fontsize=13, fontweight="bold", pad=14,
        color="#1E293B"
    )

    ax.set_xticks(x, labels)
    ax.set_ylim(0, 1.05)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.tick_params(axis="both", length=0, labelsize=10)

    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes[:, 0]:
    ax.set_ylabel("Accuracy", fontsize=11)

fig.suptitle(
    "Primacy and Recency",
    fontsize=17, fontweight="bold", color="#0F172A"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from wordfreq import top_n_list, word_frequency
from matplotlib.ticker import PercentFormatter
from wordfreq import iter_wordlist


VOWELS = set("AEIOUYÆØÅ")
LETTERS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZÆØÅ")
SIZES = [2, 3, 4,5,6]


def cv_pattern(letters):
    return "".join(
        "V" if letter in VOWELS else "C"
        for letter in letters
    )


# ============================================================
# 1. JERES ACCURACY FOR KONSONANT-/VOKALMØNSTRE
# ============================================================

serial_trials = trials[
    trials["experiment"] == "serial_recall"
].copy()

memory_rows = []

for _, trial in serial_trials.iterrows():
    sequence = [
        letter.strip().upper()
        for letter in str(trial["sequence"]).split("||")
    ]

    response = (
        []
        if pd.isna(trial["response"])
        else [
            letter.strip().upper()
            for letter in str(trial["response"]).split("||")
        ]
    )

    # Kun sekvenser med enkelte bogstaver
    if not all(letter in LETTERS for letter in sequence):
        continue

    # Rigtigt bogstav på rigtig position.
    # Manglende svar tælles som forkert.
    correct = [
        int(i < len(response) and letter == response[i])
        for i, letter in enumerate(sequence)
    ]

    for size in SIZES:
        for start in range(len(sequence) - size + 1):
            memory_rows.append({
                "length": size,
                "pattern": cv_pattern(sequence[start:start + size]),
                "accuracy": np.mean(correct[start:start + size]),
            })

memory_data = pd.DataFrame(
    memory_rows,
    columns=["length", "pattern", "accuracy"]
)

memory_summary = (
    memory_data
    .groupby(["length", "pattern"], as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        n_memory_windows=("accuracy", "count"),
    )
)


# ============================================================
# 2. MØNSTERHYPPIGHED I DANSKE ORD
#
# Tæl sammenhængende mønstre INDE I ordene.
# Hver forekomst vægtes med ordets estimerede hyppighed.
# ============================================================

danish_counts = {size: Counter() for size in SIZES}

for word in iter_wordlist("da"):
    letters = word.upper()

    # Spring tal, bindestreger og andre tegn over
    if not letters or not all(letter in LETTERS for letter in letters):
        continue

    frequency = word_frequency(word, "da")
    pattern = cv_pattern(letters)

    for size in SIZES:
        for start in range(len(pattern) - size + 1):
            part = pattern[start:start + size]
            danish_counts[size][part] += frequency

danish_rows = []

for size, counts in danish_counts.items():
    total = sum(counts.values())

    for pattern, frequency in counts.items():
        danish_rows.append({
            "length": size,
            "pattern": pattern,
            "frequency_share": frequency / total,
        })

danish_summary = pd.DataFrame(danish_rows)


# ============================================================
# 3. SAMMENLÆG DANSK HYPPIGHED OG JERES ACCURACY
# ============================================================

combined_pattern_summary = (
    memory_summary
    .merge(
        danish_summary,
        on=["length", "pattern"],
        how="left",
    )
    .sort_values(
        ["length", "frequency_share"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

combined_pattern_summary["frequency_share"] = (
    combined_pattern_summary["frequency_share"].fillna(0)
)


# ============================================================
# 4. TRE SCATTERPLOTS
# ============================================================

colors = {2: "#2563EB", 3: "#8B5CF6", 4: "#0891B2",5:"#EA580C",6 : "#DB2777"}

for size in SIZES:
    temp = combined_pattern_summary[
        combined_pattern_summary["length"] == size
    ].copy()

    if temp.empty:
        continue

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_facecolor("#F3F6FB")
    ax.set_facecolor("white")

    # Større prik = flere observationer i jeres eksperiment
    point_sizes = (
        60
        + 300
        * temp["n_memory_windows"]
        / temp["n_memory_windows"].max()
    )

    ax.scatter(
        temp["frequency_share"],
        temp["mean_accuracy"],
        s=point_sizes,
        color=colors[size],
        alpha=0.75,
        edgecolors="white",
        linewidths=1,
        zorder=3,
    )

    for _, row in temp.iterrows():
        ax.annotate(
            row["pattern"],
            (row["frequency_share"], row["mean_accuracy"]),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=11,
            fontweight="bold",
            color="#1E293B",
        )

    ax.set_title(
        f"{size}-letter C/V patterns:\n"
        "Danish frequency vs memory accuracy",
        fontsize=15,
        fontweight="bold",
        color="#0F172A",
        pad=15,
    )

    ax.set_xlabel("Pattern frequency in Danish words", fontsize=11)
    ax.set_ylabel("Mean memory accuracy", fontsize=11)

    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_ylim(0, 1.08)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.margins(x=0.15)

    ax.set_axisbelow(True)
    ax.grid(color="#E2E8F0", alpha=0.8)
    ax.tick_params(length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.ticker import PercentFormatter

conditions = ["baseline", "articulatory_suppression"]

# Kun serial recall
selected = trials[
    (trials["experiment"] == "serial_recall") &
    (trials["condition"].isin(conditions))
].copy()

# Hvert forsøg er ét datapunkt
stats = (
    selected.groupby("condition")["serial_accuracy"]
    .agg(["mean", "std", "count"])
    .reindex(conditions)
)

means = stats["mean"]

margin = (
    t.ppf(0.975, df=stats["count"] - 1)
    * stats["std"]
    / np.sqrt(stats["count"])
)

display(stats[["count"]].rename(columns={"count": "Antal datapunkter"}))


# Afgræns de viste accuracy-intervaller til 0–1
lower = (means - margin).clip(0, 1)
upper = (means + margin).clip(0, 1)

result = pd.DataFrame({
    "Accuracy": means,
    "95 % CI nedre": lower,
    "95 % CI øvre": upper,
})
result.index = ["Baseline", "Articulatory suppression"]

display(result.style.format("{:.1%}"))

# ------------------------------------------------------------
# Graf
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor("#F3F6FB")
ax.set_facecolor("white")

x = np.arange(2)

ax.bar(
    x, means.to_numpy(),
    width=0.55,
    color=["#2563EB", "#8B5CF6"],
    alpha=0.85,
    zorder=3,
)

ax.errorbar(
    x, means.to_numpy(),
    yerr=[
        (means - lower).to_numpy(),
        (upper - means).to_numpy(),
    ],
    fmt="none",
    ecolor="#1E293B",
    capsize=8,
    elinewidth=1.8,
    capthick=1.8,
    zorder=4,
)

for position, mean in zip(x, means):
    ax.text(
        position, mean / 2,
        f"{mean:.1%}",
        ha="center", va="center",
        fontsize=14, fontweight="bold", color="white",
    )

ax.set_xticks(x, ["Baseline", "Articulatory suppression"])
ax.set_ylabel("Serial recall accuracy")
ax.set_title(
    "Baseline vs Articulatory Suppression",
    fontsize=15, fontweight="bold", pad=16,
    color="#0F172A",
)

ax.set_ylim(0, 1.05)
ax.set_yticks(np.linspace(0, 1, 6))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_axisbelow(True)
ax.grid(axis="y", color="#E2E8F0")
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.tight_layout()
plt.show()